In [0]:
silver_path = "/Volumes/workspace/default/ecommerce_data/silver/"
df = spark.read.format("delta").load(silver_path)

print(f"Loaded {df.count()} rows from Silver")
df.printSchema()

Loaded 1006176 rows from Silver
root
 |-- user_id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- urban_rural: string (nullable = true)
 |-- income_level: double (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- relationship_status: string (nullable = true)
 |-- has_children: integer (nullable = true)
 |-- household_size: integer (nullable = true)
 |-- occupation: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- language_preference: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- weekly_purchases: integer (nullable = true)
 |-- monthly_spend: double (nullable = true)
 |-- cart_abandonment_rate: double (nullable = true)
 |-- review_writing_frequency: integer (nullable = true)
 |-- average_order_value: double (nullable = true)
 |-- preferred_payment_method: string (nullable = tru

In [0]:
from pyspark.sql import functions as F

country_summary = df.groupBy("country").agg(
    F.count("*").alias("total_customers"),
    F.round(F.avg("monthly_spend"), 2).alias("avg_monthly_spend"),
    F.round(F.avg("average_order_value"), 2).alias("avg_order_value"),
    F.round(F.avg("cart_abandonment_rate"), 2).alias("avg_cart_abandonment_rate"),
    F.round(F.avg("return_rate"), 2).alias("avg_return_rate"),
    F.round(F.avg("brand_loyalty_score"), 2).alias("avg_brand_loyalty_score")
).orderBy(F.desc("total_customers"))

country_summary.show(truncate=False)

+---------+---------------+-----------------+---------------+-------------------------+---------------+-----------------------+
|country  |total_customers|avg_monthly_spend|avg_order_value|avg_cart_abandonment_rate|avg_return_rate|avg_brand_loyalty_score|
+---------+---------------+-----------------+---------------+-------------------------+---------------+-----------------------+
|Brazil   |101133         |1805.66          |219.02         |44.5                     |41.45          |5.28                   |
|Canada   |100962         |3384.95          |372.54         |44.47                    |41.48          |5.28                   |
|China    |100805         |2755.34          |296.42         |44.44                    |41.64          |5.26                   |
|Germany  |100680         |3618.62          |399.36         |44.46                    |41.39          |5.27                   |
|France   |100633         |3164.32          |357.38         |44.6                     |41.52          |5

In [0]:
# NOTE: "country" added to groupBy so this table can be filtered/joined by country too
category_summary = df.groupBy("country", "product_category_preference").agg(
    F.count("*").alias("total_customers"),
    F.round(F.avg("monthly_spend"), 2).alias("avg_monthly_spend"),
    F.round(F.avg("weekly_purchases"), 2).alias("avg_weekly_purchases"),
    F.round(F.avg("coupon_usage_frequency"), 2).alias("avg_coupon_usage"),
    F.round(F.avg("purchase_conversion_rate"), 2).alias("avg_conversion_rate")
).orderBy(F.desc("total_customers"))

category_summary.show(truncate=False)

+---------+---------------------------+---------------+-----------------+--------------------+----------------+-------------------+
|country  |product_category_preference|total_customers|avg_monthly_spend|avg_weekly_purchases|avg_coupon_usage|avg_conversion_rate|
+---------+---------------------------+---------------+-----------------+--------------------+----------------+-------------------+
|India    |Electronics                |21794          |1929.43          |10.07               |2.07            |44.91              |
|Germany  |Electronics                |21782          |4481.29          |9.91                |2.07            |45.03              |
|Brazil   |Electronics                |21725          |2263.97          |10.04               |2.03            |44.74              |
|USA      |Electronics                |21724          |4804.34          |9.93                |2.09            |44.58              |
|Australia|Electronics                |21723          |4246.78          |10.

In [0]:
customer_segments = df.withColumn(
    "spend_segment",
    F.when(F.col("monthly_spend") >= 5000, "High Spender")
     .when(F.col("monthly_spend") >= 1500, "Mid Spender")
     .otherwise("Low Spender")
).withColumn(
    "loyalty_segment",
    F.when(F.col("brand_loyalty_score") >= 7, "High Loyalty")
     .when(F.col("brand_loyalty_score") >= 4, "Medium Loyalty")
     .otherwise("Low Loyalty")
)

# NOTE: "country" added to groupBy so this table can be filtered/joined by country too
segment_summary = customer_segments.groupBy("country", "spend_segment", "loyalty_segment").agg(
    F.count("*").alias("total_customers"),
    F.round(F.avg("monthly_spend"), 2).alias("avg_monthly_spend"),
    F.round(F.avg("return_rate"), 2).alias("avg_return_rate")
).orderBy(F.desc("total_customers"))

segment_summary.show(truncate=False)

+---------+-------------+---------------+---------------+-----------------+---------------+
|country  |spend_segment|loyalty_segment|total_customers|avg_monthly_spend|avg_return_rate|
+---------+-------------+---------------+---------------+-----------------+---------------+
|India    |Low Spender  |Low Loyalty    |23246          |692.27           |41.97          |
|India    |Low Spender  |High Loyalty   |22289          |698.74           |35.53          |
|Brazil   |Low Spender  |Low Loyalty    |20657          |711.88           |41.8           |
|Brazil   |Low Spender  |High Loyalty   |19876          |711.91           |35.38          |
|China    |Mid Spender  |Low Loyalty    |17729          |2863.4           |44.39          |
|France   |Mid Spender  |Low Loyalty    |17428          |2937.42          |44.23          |
|Japan    |Mid Spender  |Low Loyalty    |17281          |2921.13          |43.57          |
|Canada   |Mid Spender  |Low Loyalty    |17204          |2974.89          |43.71

In [0]:
# NOTE: "country" added to groupBy so this table can be filtered/joined by country too
device_payment_summary = df.groupBy("country", "device_type", "preferred_payment_method").agg(
    F.count("*").alias("total_customers"),
    F.round(F.avg("average_order_value"), 2).alias("avg_order_value"),
    F.round(F.avg("daily_session_time_minutes"), 2).alias("avg_session_time_minutes"),
    F.round(F.avg("purchase_conversion_rate"), 2).alias("avg_conversion_rate")
).orderBy(F.desc("total_customers"))

device_payment_summary.show(truncate=False)

+---------+-----------+------------------------+---------------+---------------+------------------------+-------------------+
|country  |device_type|preferred_payment_method|total_customers|avg_order_value|avg_session_time_minutes|avg_conversion_rate|
+---------+-----------+------------------------+---------------+---------------+------------------------+-------------------+
|India    |Mobile     |Google Pay              |25387          |188.39         |63.65                   |51.12              |
|China    |Mobile     |Google Pay              |22773          |296.01         |64.1                    |51.19              |
|Germany  |Mobile     |Bank Transfer           |20255          |398.97         |63.9                    |50.9               |
|USA      |Mobile     |Credit Card             |20010          |426.1          |63.67                   |50.95              |
|Japan    |Mobile     |Credit Card             |19645          |347.32         |64.23                   |50.86        

In [0]:
age_group_df = df.withColumn(
    "age_group",
    F.when(F.col("age") < 25, "18-24")
     .when(F.col("age") < 35, "25-34")
     .when(F.col("age") < 45, "35-44")
     .when(F.col("age") < 55, "45-54")
     .when(F.col("age") < 65, "55-64")
     .otherwise("65+")
)

# NOTE: "country" added to groupBy so this table can be filtered/joined by country too
age_group_summary = age_group_df.groupBy("country", "age_group", "gender").agg(
    F.count("*").alias("total_customers"),
    F.round(F.avg("monthly_spend"), 2).alias("avg_monthly_spend"),
    F.round(F.avg("overall_stress_level"), 2).alias("avg_stress_level"),
    F.round(F.avg("mental_health_score"), 2).alias("avg_mental_health_score")
).orderBy("age_group", "gender")

age_group_summary.show(truncate=False)

+---------+---------+------+---------------+-----------------+----------------+-----------------------+
|country  |age_group|gender|total_customers|avg_monthly_spend|avg_stress_level|avg_mental_health_score|
+---------+---------+------+---------------+-----------------+----------------+-----------------------+
|China    |18-24    |Female|4994           |1488.71          |5.02            |5.01                   |
|UK       |18-24    |Female|5084           |1968.57          |4.99            |5.03                   |
|France   |18-24    |Female|5149           |1769.75          |5.01            |4.93                   |
|Canada   |18-24    |Female|5203           |1923.06          |4.99            |4.96                   |
|USA      |18-24    |Female|4973           |2201.11          |5.06            |4.95                   |
|Brazil   |18-24    |Female|5144           |966.82           |4.96            |5.03                   |
|Australia|18-24    |Female|5073           |1891.27          |5.

In [0]:
gold_base = "/Volumes/workspace/default/ecommerce_data/gold/"

tables = {
    "country_summary": country_summary,
    "category_summary": category_summary,
    "customer_segment_summary": segment_summary,
    "device_payment_summary": device_payment_summary,
    "age_gender_summary": age_group_summary
}

for name, table_df in tables.items():
    path = gold_base + name
    dbutils.fs.rm(path, recurse=True)
    table_df.write.format("delta").mode("overwrite").save(path)
    print(f"{name}: {table_df.count()} rows written to {path}")

country_summary: 10 rows written to /Volumes/workspace/default/ecommerce_data/gold/country_summary
category_summary: 90 rows written to /Volumes/workspace/default/ecommerce_data/gold/category_summary
customer_segment_summary: 90 rows written to /Volumes/workspace/default/ecommerce_data/gold/customer_segment_summary
device_payment_summary: 280 rows written to /Volumes/workspace/default/ecommerce_data/gold/device_payment_summary
age_gender_summary: 300 rows written to /Volumes/workspace/default/ecommerce_data/gold/age_gender_summary


In [0]:
gold_base = "/Volumes/workspace/default/ecommerce_data/gold/"

country_summary = spark.read.format("delta").load(gold_base + "country_summary")
display(country_summary)

country,total_customers,avg_monthly_spend,avg_order_value,avg_cart_abandonment_rate,avg_return_rate,avg_brand_loyalty_score
Brazil,101133,1805.66,219.02,44.5,41.45,5.28
Canada,100962,3384.95,372.54,44.47,41.48,5.28
China,100805,2755.34,296.42,44.44,41.64,5.26
Germany,100680,3618.62,399.36,44.46,41.39,5.27
France,100633,3164.32,357.38,44.6,41.52,5.26
Australia,100597,3396.7,372.75,44.49,41.37,5.28
USA,100584,3942.83,425.36,44.5,41.54,5.27
India,100545,1531.93,187.28,44.47,41.42,5.3
UK,100467,3515.86,386.49,44.48,41.61,5.27
Japan,99770,3021.95,344.17,44.53,41.44,5.25


In [0]:
category_summary = spark.read.format("delta").load(gold_base + "category_summary")
display(category_summary)

country,product_category_preference,total_customers,avg_monthly_spend,avg_weekly_purchases,avg_coupon_usage,avg_conversion_rate
India,Electronics,21794,1929.43,10.07,2.07,44.91
Germany,Electronics,21782,4481.29,9.91,2.07,45.03
Brazil,Electronics,21725,2263.97,10.04,2.03,44.74
USA,Electronics,21724,4804.34,9.93,2.09,44.58
Australia,Electronics,21723,4246.78,10.0,2.08,44.43
China,Electronics,21718,3457.0,10.04,2.06,44.99
Canada,Electronics,21547,4224.77,10.0,2.1,45.15
Japan,Electronics,21525,3748.64,10.01,2.06,44.8
France,Electronics,21423,3950.89,9.99,2.04,45.4
UK,Electronics,21374,4323.73,10.01,2.06,44.74


In [0]:
customer_segment_summary = spark.read.format("delta").load(gold_base + "customer_segment_summary")
display(customer_segment_summary)

country,spend_segment,loyalty_segment,total_customers,avg_monthly_spend,avg_return_rate
India,Low Spender,Low Loyalty,23246,692.27,41.97
India,Low Spender,High Loyalty,22289,698.74,35.53
Brazil,Low Spender,Low Loyalty,20657,711.88,41.8
Brazil,Low Spender,High Loyalty,19876,711.91,35.38
China,Mid Spender,Low Loyalty,17729,2863.4,44.39
France,Mid Spender,Low Loyalty,17428,2937.42,44.23
Japan,Mid Spender,Low Loyalty,17281,2921.13,43.57
Canada,Mid Spender,Low Loyalty,17204,2974.89,43.71
UK,Mid Spender,Low Loyalty,17100,2996.58,43.45
Australia,Mid Spender,Low Loyalty,17091,2982.05,43.09


In [0]:
device_payment_summary = spark.read.format("delta").load(gold_base + "device_payment_summary")
display(device_payment_summary)

country,device_type,preferred_payment_method,total_customers,avg_order_value,avg_session_time_minutes,avg_conversion_rate
India,Mobile,Google Pay,25387,188.39,63.65,51.12
China,Mobile,Google Pay,22773,296.01,64.1,51.19
Germany,Mobile,Bank Transfer,20255,398.97,63.9,50.9
USA,Mobile,Credit Card,20010,426.1,63.67,50.95
Japan,Mobile,Credit Card,19645,347.32,64.23,50.86
Canada,Mobile,Credit Card,18514,373.48,64.2,50.94
France,Mobile,Bank Transfer,17288,356.2,63.96,51.01
UK,Mobile,Debit Card,17257,385.83,63.91,51.2
Brazil,Mobile,Debit Card,17199,219.42,64.03,50.77
Australia,Mobile,Credit Card,15877,371.33,64.38,51.09


In [0]:
age_gender_summary = spark.read.format("delta").load(gold_base + "age_gender_summary")
display(age_gender_summary)

country,age_group,gender,total_customers,avg_monthly_spend,avg_stress_level,avg_mental_health_score
USA,18-24,Female,4973,2201.11,5.06,4.95
Germany,18-24,Female,5006,2006.35,5.04,5.02
China,18-24,Female,4994,1488.71,5.02,5.01
Australia,18-24,Female,5073,1891.27,5.03,4.94
UK,18-24,Female,5084,1968.57,4.99,5.03
Canada,18-24,Female,5203,1923.06,4.99,4.96
Brazil,18-24,Female,5144,966.82,4.96,5.03
France,18-24,Female,5149,1769.75,5.01,4.93
India,18-24,Female,5046,834.7,5.01,5.0
Japan,18-24,Female,5053,1699.88,5.02,5.02
